In [8]:
import ipywidgets as widgets
print(widgets.__version__)

8.1.8


In [9]:
!pip install nltk scikit-learn pdfminer.six python-docx ipywidgets

import re
import string
import nltk
import numpy as np
import ipywidgets as widgets
import io
import base64
from IPython.display import display, HTML, clear_output
from sklearn.feature_extraction.text import TfidfVectorizer   # ← TF-IDF, better than CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pdfminer.high_level import extract_text_to_fp
from pdfminer.layout import LAParams
from docx import Document


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# ─── SKILLS DATABASE (expanded from 13 → 60+) ───
SKILLS_DB = {
    "Languages":    ["python","java","c++","c#","javascript","typescript","r","scala","go","rust","kotlin","swift"],
    "ML/AI":        ["machine learning","deep learning","nlp","computer vision","reinforcement learning",
                     "transformers","llm","generative ai","bert","gpt","yolo","opencv"],
    "Frameworks":   ["tensorflow","pytorch","keras","scikit-learn","xgboost","lightgbm","huggingface",
                     "flask","fastapi","django","streamlit","gradio","react","nodejs"],
    "Data":         ["sql","pandas","numpy","spark","hadoop","tableau","power bi","excel",
                     "data analysis","data visualization","etl","dbt"],
    "Cloud/MLOps":  ["aws","azure","gcp","docker","kubernetes","mlflow","airflow","ci/cd","git","linux"],
    "Soft Skills":  ["communication","leadership","teamwork","problem solving","agile","scrum"],
}
ALL_SKILLS = [s for group in SKILLS_DB.values() for s in group]

# ─── TEXT EXTRACTION ───
def extract_pdf_text(file_bytes: bytes) -> str:
    output = io.StringIO()
    extract_text_to_fp(io.BytesIO(file_bytes), output, laparams=LAParams())
    return output.getvalue()

def extract_docx_text(file_bytes: bytes) -> str:
    doc = Document(io.BytesIO(file_bytes))
    return "\n".join([p.text for p in doc.paragraphs])

def extract_text_from_bytes(file_bytes: bytes, filename: str) -> str:
    ext = filename.rsplit(".", 1)[-1].lower()
    if ext == "pdf":
        return extract_pdf_text(file_bytes)
    elif ext == "docx":
        return extract_docx_text(file_bytes)
    return ""

# ─── CLEANING ───
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = [w for w in text.split() if w not in stop_words and len(w) > 1]
    return " ".join(words)

# ─── SIMILARITY (TF-IDF) ───
def calculate_similarity(resume_clean: str, jd_clean: str) -> float:
    vec = TfidfVectorizer(ngram_range=(1, 2))  # bigrams too e.g. "machine learning"
    matrix = vec.fit_transform([resume_clean, jd_clean])
    score = cosine_similarity(matrix)[0][1]
    return round(score * 100, 2)

# ─── SKILL EXTRACTION ───
def extract_skills(text: str) -> dict:
    found = {}
    for category, skills in SKILLS_DB.items():
        matched = [s for s in skills if s in text]
        if matched:
            found[category] = matched
    return found

def get_missing_skills(resume_text: str, jd_text: str) -> list:
    jd_skills   = [s for s in ALL_SKILLS if s in jd_text]
    res_skills  = [s for s in ALL_SKILLS if s in resume_text]
    return list(set(jd_skills) - set(res_skills))

# ─── FEEDBACK ───
def get_feedback(score: float, missing: list) -> tuple:
    if score >= 75:
        level, emoji, color = "Strong match", "✅", "#28a745"
        advice = "You're well-aligned. Tailor your summary section to echo JD keywords."
    elif score >= 50:
        level, emoji, color = "Moderate match", "⚠️", "#ffc107"
        advice = f"Add these skills to your resume if you have them: {', '.join(missing[:5]) or 'none found'}."
    else:
        level, emoji, color = "Weak match", "❌", "#dc3545"
        advice = "Significant gaps. Consider upskilling or targeting a better-fit role."
    return level, emoji, color, advice

# ─── HTML REPORT ───
def build_report_html(score, resume_skills, missing, level, emoji, color, advice):
    # Skill pills per category
    skill_html = ""
    for cat, skills in resume_skills.items():
        pills = "".join(f'<span style="background:#e8f5e9;color:#2e7d32;padding:3px 10px;border-radius:999px;font-size:12px;margin:3px;display:inline-block">{s}</span>' for s in skills)
        skill_html += f'<p style="font-size:12px;color:#888;margin:8px 0 2px">{cat}</p>{pills}'

    miss_pills = "".join(
        f'<span style="background:#fdecea;color:#c62828;padding:3px 10px;border-radius:999px;font-size:12px;margin:3px;display:inline-block">{s}</span>'
        for s in missing
    ) or '<span style="color:#888;font-size:13px">None — great coverage!</span>'

    bar_color = "#28a745" if score >= 75 else ("#ffc107" if score >= 50 else "#dc3545")

    return f"""
    <div style="font-family:sans-serif;max-width:700px;margin:16px 0">
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px;margin-bottom:16px">
        <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:12px">
          <span style="font-size:18px;font-weight:600">Resume Match Score</span>
          <span style="font-size:28px;font-weight:700;color:{bar_color}">{score}%</span>
        </div>
        <div style="background:#f0f0f0;border-radius:999px;height:10px;width:100%">
          <div style="background:{bar_color};width:{score}%;height:10px;border-radius:999px;transition:width 1s"></div>
        </div>
        <div style="margin-top:12px;padding:10px 14px;background:#f9f9f9;border-radius:8px;border-left:4px solid {color}">
          <span style="font-size:15px">{emoji} <b>{level}</b> — {advice}</span>
        </div>
      </div>
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px;margin-bottom:16px">
        <div style="font-size:16px;font-weight:600;margin-bottom:12px">🧠 Skills Found in Resume</div>
        {skill_html or '<span style="color:#888;font-size:13px">No known skills detected</span>'}
      </div>
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px">
        <div style="font-size:16px;font-weight:600;margin-bottom:12px">⚠️ Skills Missing (in JD but not resume)</div>
        {miss_pills}
      </div>
    </div>
    """

In [11]:
# ─── WIDGETS ───
uploader  = widgets.FileUpload(accept='.pdf,.docx', multiple=False, description='📄 Upload Resume')
jd_box    = widgets.Textarea(placeholder='Paste the Job Description here...', layout=widgets.Layout(width='100%', height='140px'))
run_btn   = widgets.Button(description='🔍 Analyze Resume', button_style='primary', layout=widgets.Layout(margin='10px 0'))
out       = widgets.Output()

display(widgets.HTML("<h2>🤖 AI Resume Analyzer</h2>"))
display(widgets.HTML("<p>Step 1 → Upload resume (PDF/DOCX) &nbsp;|&nbsp; Step 2 → Paste job description &nbsp;|&nbsp; Step 3 → Analyze</p>"))
display(uploader)
display(widgets.HTML("<b>Job Description:</b>"))
display(jd_box)
display(run_btn)
display(out)

def on_analyze(b):
    with out:
        clear_output()

        # ── read uploaded file ──
        if not uploader.value:
            print("❌ Please upload a resume first.")
            return
        try:
            if isinstance(uploader.value, tuple):
                f        = uploader.value[0]
                filename = f['name']
                content  = bytes(f['content'])
            else:
                filename = list(uploader.value.keys())[0]
                content  = bytes(uploader.value[filename]['content'])
        except Exception as e:
            print(f"❌ File read error: {e}"); return

        jd_raw = jd_box.value.strip()
        if not jd_raw:
            print("❌ Please paste a job description."); return

        print("⏳ Analyzing...")

        # ── process ──
        raw_resume  = extract_text_from_bytes(content, filename)
        if not raw_resume.strip():
            print("❌ Could not extract text from resume. Is it a scanned image PDF?"); return

        res_clean   = clean_text(raw_resume)
        jd_clean    = clean_text(jd_raw)
        score       = calculate_similarity(res_clean, jd_clean)
        res_skills  = extract_skills(res_clean)
        missing     = get_missing_skills(res_clean, jd_clean)
        level, emoji, color, advice = get_feedback(score, missing)

        display(HTML(build_report_html(score, res_skills, missing, level, emoji, color, advice)))

run_btn.on_click(on_analyze)

HTML(value='<h2>🤖 AI Resume Analyzer</h2>')

HTML(value='<p>Step 1 → Upload resume (PDF/DOCX) &nbsp;|&nbsp; Step 2 → Paste job description &nbsp;|&nbsp; St…

FileUpload(value=(), accept='.pdf,.docx', description='📄 Upload Resume')

HTML(value='<b>Job Description:</b>')

Textarea(value='', layout=Layout(height='140px', width='100%'), placeholder='Paste the Job Description here...…

Button(button_style='primary', description='🔍 Analyze Resume', layout=Layout(margin='10px 0'), style=ButtonSty…

Output()

8.1.8
